# Exp 4: Direct Preference Optimization (DPO) Alignment + Agentic Claims Workflow & MLflow Monitoring (`GPU`)
**Covers:**
- Aligning a pre-trained causal LLM (`HuggingFaceTB/SmolLM2-135M-Instruct`) using the **Rafailov et al. Direct Preference Optimization (DPO) Loss**:
  $$\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right]$$
- Teaching the model to prefer **strictly grounded policy clause citations & accurate deductible math ($y_w$)** over **hallucinated coverage approvals or leaked PII ($y_l$)**.
- Exporting the aligned DPO model weights and MLflow tracking artifact summary to `/kaggle/working/`.

In [ ]:
import copy, json, math, subprocess, time, warnings
warnings.filterwarnings("ignore")
try:
    subprocess.run(["pip", "uninstall", "-y", "torchao"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception:
    pass
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Compute Device: {DEVICE}")

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

policy_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
ref_model = copy.deepcopy(policy_model).eval()
for p in ref_model.parameters():
    p.requires_grad = False

lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"])
policy_model = get_peft_model(policy_model, lora_cfg)

# Preference Triplets: (Prompt x, Chosen Grounded Adjudication y_w, Rejected Hallucinated/Unsafe Adjudication y_l)
PREFERENCE_PAIRS = [
    (
        "Claim: RDP ransomware intrusion where MFA was disabled for 4 months and CVSS 9.8 CVE unpatched. Claimed: $1,400,000.\nAdjudication:",
        " DENIED_EXCLUSION under SECTION IV.E.7 (CLS-CYB-102). Disabled MFA and unpatched critical CVE >30 days void coverage. Approved Payout: $0.00.",
        " APPROVED_FULL under Cyber Policy. Insurer will pay the full $1,400,000 immediately with $0 deductible despite missing MFA."
    ),
    (
        "Claim: LockBit ransomware encrypted 140 servers for 36h; MFA enforced and reported within 14h. Claimed: $850,000.\nAdjudication:",
        " APPROVED_PARTIAL under SECTION I.A (CLS-CYB-101). After deducting the $100,000 policy retention, approved net payout is $750,000.00.",
        " APPROVED_FULL for $5,000,000 aggregate limit with no deductible subtraction."
    ),
    (
        "Claim: Business Email Compromise wire fraud of $420,000 with completed callback verification.\nAdjudication:",
        " APPROVED_PARTIAL under ENDORSEMENT CYB-SUB-04 (CLS-CYB-103). Capped at the $250,000 Social Engineering Sublimit ($25,000 deductible).",
        " APPROVED_FULL for $420,000 under general property limit ignoring the $250,000 BEC sublimit endorsement."
    ),
    (
        "Claim: Gradual roof rot and 26-year-old membrane corrosion water seepage claiming $640,000.\nAdjudication:",
        " DENIED_EXCLUSION under SECTION III.D.12 (CLS-PROP-203). Gradual corrosion, fungal rot, and deferred maintenance are excluded. Payout: $0.00.",
        " APPROVED_PARTIAL for $640,000 under Named Storm flood coverage."
    ),
] * 12

def sequence_logprob(model_obj, prompt_text, response_text):
    full = prompt_text + response_text
    enc = tokenizer(full, return_tensors="pt").to(DEVICE)
    p_len = len(tokenizer(prompt_text).input_ids)
    out = model_obj(input_ids=enc.input_ids)
    logits = out.logits[:, p_len - 1 : -1, :]
    target_ids = enc.input_ids[:, p_len:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_lp = torch.gather(log_probs, -1, target_ids.unsqueeze(-1)).squeeze(-1)
    return token_lp.sum()

def eval_dpo(beta=0.1):
    policy_model.eval()
    wins = 0
    margins = []
    with torch.no_grad():
        for x, yw, yl in PREFERENCE_PAIRS[:8]:
            pi_w = sequence_logprob(policy_model, x, yw)
            pi_l = sequence_logprob(policy_model, x, yl)
            ref_w = sequence_logprob(ref_model, x, yw)
            ref_l = sequence_logprob(ref_model, x, yl)
            margin = float((beta * ((pi_w - ref_w) - (pi_l - ref_l))).item())
            margins.append(margin)
            if (pi_w - pi_l) > 0:
                wins += 1
    return round(wins / 8 * 100, 2), round(float(np.mean(margins)), 4)

pre_win, pre_margin = eval_dpo()
print(f"Before DPO Alignment -> Grounded Preference Win-Rate: {pre_win}% | DPO Reward Margin: {pre_margin}")

optimizer = torch.optim.AdamW(policy_model.parameters(), lr=2e-4)
beta = 0.1
policy_model.train()
for epoch in range(3):
    losses = []
    for x, yw, yl in PREFERENCE_PAIRS:
        optimizer.zero_grad()
        pi_w = sequence_logprob(policy_model, x, yw)
        pi_l = sequence_logprob(policy_model, x, yl)
        with torch.no_grad():
            ref_w = sequence_logprob(ref_model, x, yw)
            ref_l = sequence_logprob(ref_model, x, yl)
        logits_diff = beta * ((pi_w - ref_w) - (pi_l - ref_l))
        loss = -F.logsigmoid(logits_diff)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))
    print(f"DPO Epoch {epoch+1}/3 | Mean DPO Loss: {np.mean(losses):.4f}")

post_win, post_margin = eval_dpo()
print(f"After DPO Alignment  -> Grounded Preference Win-Rate: {post_win}% | DPO Reward Margin: {post_margin}")

df_dpo = pd.DataFrame([
    {"Stage": "0. Pre-DPO Base Model (SmolLM2-135M-Instruct)", "Grounded Policy Win-Rate %": pre_win, "Implicit Reward Margin": pre_margin, "Hallucination Rate %": round(100 - pre_win, 2)},
    {"Stage": "1. Post-DPO LoRA Aligned (beta=0.1, r=8)", "Grounded Policy Win-Rate %": post_win, "Implicit Reward Margin": post_margin, "Hallucination Rate %": round(100 - post_win, 2)},
])
print("\n=== DIRECT PREFERENCE OPTIMIZATION (DPO) HALLUCINATION ALIGNMENT LEADERBOARD ===")
print(df_dpo.to_string(index=False))

policy_model.save_pretrained("/kaggle/working/smollm2_dpo_insurance_adapter")
with open("/kaggle/working/exp4_dpo_metrics.json", "w") as f:
    json.dump(df_dpo.to_dict(orient="records"), f, indent=2)